# FRAML AI — Inference Only (vast.ai)

Downloads a pre-trained GGUF from HuggingFace Hub, registers it with Ollama, and starts serving.
No training required.

**Pipeline:**
1. Download GGUF from HF Hub
2. Install Ollama
3. Write Modelfile (Qwen 2.5 tool-calling template)
4. Register model + start server
5. Smoke test

**Recommended vast.ai instance:** RTX 3090 24GB, **20GB disk** is enough (GGUF is ~4.7 GB)

## Cell 1 — Configuration

In [ ]:
# ── Which model to pull ────────────────────────────────────────────────────
HF_REPO       = "speri420/qwen-framl-v12"      # change to v13 when ready
HF_FILENAME   = "qwen-framl-q4km.gguf"
HF_TOKEN      = ""                             # leave blank — repo is public

GGUF_PATH     = f"/workspace/{HF_FILENAME}"
MODELFILE_PATH = "/tmp/Modelfile.framl"
OLLAMA_MODEL  = "qwen-framl"

print(f"Model repo : {HF_REPO}")
print(f"GGUF path  : {GGUF_PATH}")
print(f"Ollama name: {OLLAMA_MODEL}")

## Cell 2 — Download GGUF from HuggingFace Hub

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

from huggingface_hub import hf_hub_download
from pathlib import Path

gguf = Path(GGUF_PATH)

if gguf.exists() and gguf.stat().st_size > 1_000_000_000:
    print(f"GGUF already present: {gguf}  ({gguf.stat().st_size / 1e9:.1f} GB) — skipping download.")
else:
    print(f"Downloading {HF_REPO}/{HF_FILENAME} → {GGUF_PATH} ...")
    kwargs = dict(
        repo_id=HF_REPO,
        filename=HF_FILENAME,
        local_dir="/workspace",
    )
    if HF_TOKEN:
        kwargs["token"] = HF_TOKEN
    hf_hub_download(**kwargs)
    print(f"Download complete: {gguf.stat().st_size / 1e9:.1f} GB")

## Cell 3 — Install Ollama

In [ ]:
import shutil, subprocess

if shutil.which("ollama") is None:
    print("Installing Ollama ...")
    result = subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else "(no stdout)")
    if result.returncode != 0:
        print("STDERR:", result.stderr[-1000:])
    else:
        print("Ollama installed.")
else:
    v = subprocess.run(["ollama", "--version"], capture_output=True, text=True)
    print(f"Ollama already installed: {v.stdout.strip()}")

## Cell 4 — Write Modelfile

**Critical:** The full Qwen 2.5 TEMPLATE block must be present — without it Ollama marks the model
as *does not support tools* and every tool call returns a 400 error.

`temperature=0.0` uses greedy decoding — more deterministic tool routing.

In [ ]:
from pathlib import Path

modelfile = (
    f"FROM {GGUF_PATH}\n"
    "\n"
    "TEMPLATE \"\"\"\n"
    "{{- if or .System .Tools }}<|im_start|>system\n"
    "{{- if .System }}\n"
    "{{ .System }}\n"
    "{{- end }}\n"
    "{{- if .Tools }}\n"
    "\n"
    "# Tools\n"
    "\n"
    "You may call one or more functions to assist with the user query.\n"
    "\n"
    "You are provided with function signatures within <tools></tools> XML tags:\n"
    "<tools>\n"
    "{{- range .Tools }}\n"
    "{\"type\": \"function\", \"function\": {{ .Function }}}\n"
    "{{- end }}\n"
    "</tools>\n"
    "\n"
    "For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n"
    "<tool_call>\n"
    "{\"name\": <function-name>, \"arguments\": <args-json-object>}\n"
    "</tool_call>\n"
    "{{- end }}<|im_end|>\n"
    "{{ end }}\n"
    "{{- range $i, $_ := .Messages }}\n"
    "{{- $last := eq (len (slice $.Messages $i)) 1 }}\n"
    "{{- if eq .Role \"user\" }}<|im_start|>user\n"
    "{{ .Content }}<|im_end|>\n"
    "{{ else if eq .Role \"assistant\" }}<|im_start|>assistant\n"
    "{{ if .Content }}{{ .Content }}\n"
    "{{- else if .ToolCalls }}<tool_call>\n"
    "{{ range .ToolCalls }}{\"name\": \"{{ .Function.Name }}\", \"arguments\": {{ .Function.Arguments }}}\n"
    "{{ end }}</tool_call>\n"
    "{{- end }}{{ if not $last }}<|im_end|>\n"
    "{{ end }}\n"
    "{{- else if eq .Role \"tool\" }}<|im_start|>user\n"
    "<tool_response>\n"
    "{{ .Content }}\n"
    "</tool_response><|im_end|>\n"
    "{{ end }}\n"
    "{{- end }}<|im_start|>assistant\n"
    "\"\"\"\n"
    "\n"
    "PARAMETER num_ctx 4096\n"
    "PARAMETER num_predict 1024\n"
    "PARAMETER temperature 0.0\n"
    "PARAMETER top_p 0.9\n"
    "PARAMETER stop \"<|im_end|>\"\n"
    "\n"
    "SYSTEM \"You are a FRAML (Fraud + AML) analytics AI assistant. "
    "You analyze false positive/false negative trade-offs in AML alert thresholds, "
    "perform customer behavioral segmentation, and interpret clustering results. "
    "Use the available tools to retrieve data, then provide clear, analytical insights. "
    "Be concise and reference specific numbers when interpreting results.\"\n"
)

Path(MODELFILE_PATH).write_text(modelfile)
print(f"Modelfile written to {MODELFILE_PATH}")
print("--- Preview (first 20 lines) ---")
print("\n".join(modelfile.splitlines()[:20]))

## Cell 5 — Start Ollama Server

⚠️ **Run this cell, then start Ollama from the vast.ai terminal instead if the process dies:**
```bash
OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > /tmp/ollama.log 2>&1 &
```

In [ ]:
import os, subprocess, time

# Kill any existing Ollama instance
subprocess.run("pkill -f 'ollama serve' || true", shell=True)
time.sleep(2)

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0:11434"

subprocess.Popen(
    ["ollama", "serve"],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("Waiting for Ollama to start ...")
time.sleep(6)

check = subprocess.run(["curl", "-s", "http://localhost:11434/api/tags"],
                       capture_output=True, text=True)
if check.returncode == 0 and "models" in check.stdout:
    print("Ollama is up.")
else:
    print("Ollama may still be starting — wait a few seconds and re-run, or start from terminal.")

## Cell 6 — Register Model with Ollama

In [ ]:
import subprocess

print(f"Registering '{OLLAMA_MODEL}' from {MODELFILE_PATH} ...")
result = subprocess.run(
    ["ollama", "create", OLLAMA_MODEL, "-f", MODELFILE_PATH],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)
else:
    print(f"Model '{OLLAMA_MODEL}' registered successfully.")

# Confirm it appears in the list
lst = subprocess.run(["ollama", "list"], capture_output=True, text=True)
print(lst.stdout)

## Cell 7 — Smoke Test

Sends a tool-calling request. Expects the model to respond with a `threshold_tuning` tool call — not plain text.

In [ ]:
import urllib.request, json

payload = json.dumps({
    "model": OLLAMA_MODEL,
    "messages": [
        {"role": "user", "content": "Show FP/FN trade-off for Business customers"}
    ],
    "stream": False,
    "tools": [
        {
            "type": "function",
            "function": {
                "name": "threshold_tuning",
                "description": "Compute FP/FN trade-off across threshold range",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "segment": {"type": "string", "enum": ["Business", "Individual"]},
                        "threshold_column": {"type": "string"}
                    },
                    "required": ["segment"]
                }
            }
        }
    ]
}).encode()

req = urllib.request.Request(
    "http://localhost:11434/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"}
)

with urllib.request.urlopen(req, timeout=90) as resp:
    result = json.loads(resp.read())

msg = result["choices"][0]["message"]
if msg.get("tool_calls"):
    tc = msg["tool_calls"][0]
    print(f"✓ Tool called : {tc['function']['name']}")
    print(f"  Arguments   : {tc['function']['arguments']}")
    print("\nSMOKE TEST PASSED")
else:
    print("✗ WARNING: model returned text instead of a tool call")
    print(msg.get("content", ""))

## Cell 8 — Regression Test (prompts 3, 4, 5)

These are the V12 partial failures — model should call `rule_sar_backtest` directly, NOT `list_rules` first.

In [ ]:
import urllib.request, json

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "rule_sar_backtest",
            "description": "Run SAR backtest for a specific AML rule",
            "parameters": {
                "type": "object",
                "properties": {
                    "risk_factor": {"type": "string"},
                    "sweep_param": {"type": "string"}
                },
                "required": ["risk_factor"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_rules",
            "description": "List all available AML rules",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

SYSTEM = (
    "You are a FRAML threshold tuning specialist. "
    "For any named-rule SAR backtest request, call rule_sar_backtest immediately. "
    "Do NOT call list_rules first."
)

test_prompts = [
    ("Show SAR backtest for Activity Deviation ACH rule",   "rule_sar_backtest"),
    ("SAR backtest for Activity Deviation Check",           "rule_sar_backtest"),
    ("What is the SAR catch rate for the Detect Excessive rule?", "rule_sar_backtest"),
    ("Run a SAR backtest for Elder Abuse",                  "rule_sar_backtest"),
    ("SAR backtest for Velocity Single",                    "rule_sar_backtest"),
]

def call_model(user_msg):
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user",   "content": user_msg}
        ],
        "stream": False,
        "tools": TOOLS
    }).encode()
    req = urllib.request.Request(
        "http://localhost:11434/v1/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    with urllib.request.urlopen(req, timeout=90) as resp:
        return json.loads(resp.read())

print(f"Testing {len(test_prompts)} prompts...\n")
passed = 0
for prompt, expected_tool in test_prompts:
    result = call_model(prompt)
    msg = result["choices"][0]["message"]
    if msg.get("tool_calls"):
        actual = msg["tool_calls"][0]["function"]["name"]
        args   = msg["tool_calls"][0]["function"]["arguments"]
        ok = actual == expected_tool
        status = "PASS" if ok else "FAIL"
        if ok:
            passed += 1
        print(f"[{status}] {prompt[:60]}")
        print(f"       → {actual}({args})")
    else:
        print(f"[FAIL] {prompt[:60]}")
        print(f"       → no tool call: {msg.get('content','')[:80]}")
    print()

print(f"Result: {passed}/{len(test_prompts)} passed")

## Cell 9 — Connect Dash App

### Option A — Direct IP (if port 11434 is exposed in vast.ai)
```powershell
$env:OLLAMA_BASE_URL = "http://<vast-public-ip>:<mapped-port>/v1"
$env:OLLAMA_MODEL    = "qwen-framl"
.venv\Scripts\python.exe application.py
```

### Option B — Cloudflare tunnel (no port expose needed)
In vast.ai UI → Advanced Connection Options → new tunnel on port **11434**
```powershell
$env:OLLAMA_BASE_URL = "https://<tunnel-url>/v1"
$env:OLLAMA_MODEL    = "qwen-framl"
.venv\Scripts\python.exe application.py
```

### Keep Ollama alive after closing notebook
Run from vast.ai terminal:
```bash
OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > /tmp/ollama.log 2>&1 &
ollama list
```

---
### Swap to V13 when ready
Change Cell 1:
```python
HF_REPO = "speri420/qwen-framl-v13"
```
Then re-run Cells 2, 6, 7, 8.